> **MIT License** - Copyright (c) 2026 QuantWall. Full text: https://opensource.org/license/mit
>
> **Disclaimer.** This content is general information and software-engineering education only. It is not financial product advice, a recommendation, or an offer to buy or sell any financial product, and it does not take into account your objectives, financial situation or needs. The author does not hold an Australian Financial Services Licence and is not a registered Commodity Trading Advisor. Trading futures and leveraged products carries a high risk of loss. Consult a licensed adviser before making any financial decision.
>
> **HYPOTHETICAL OR SIMULATED PERFORMANCE RESULTS HAVE CERTAIN LIMITATIONS. UNLIKE AN ACTUAL PERFORMANCE RECORD, SIMULATED RESULTS DO NOT REPRESENT ACTUAL TRADING. ALSO, SINCE THE TRADES HAVE NOT BEEN EXECUTED, THE RESULTS MAY HAVE UNDER-OR-OVER COMPENSATED FOR THE IMPACT, IF ANY, OF CERTAIN MARKET FACTORS, SUCH AS LACK OF LIQUIDITY. SIMULATED TRADING PROGRAMS IN GENERAL ARE ALSO SUBJECT TO THE FACT THAT THEY ARE DESIGNED WITH THE BENEFIT OF HINDSIGHT. NO REPRESENTATION IS BEING MADE THAT ANY ACCOUNT WILL OR IS LIKELY TO ACHIEVE PROFIT OR LOSSES SIMILAR TO THOSE SHOWN.**
>
> All data shown is synthetic or publicly available sample data. Features are standard textbook baselines (lagged returns, ATR, RSI, moving averages, volume). No proprietary signal is disclosed or implied.

# Why systematic and statistical: every number in the video

Each cell reproduces one claim from the video, from a formula or a seeded simulation of pure noise. Change the inputs and see how the conclusions move. Requirements: `numpy`.

In [ ]:
import numpy as np

## 1. Six charts, three "patterns", no edge
Six random walks: ±1% a day, no drift, 250 days, seed 179. Code picks the steepest fitted slope ("uptrend"), the most negative one ("sliding"), and the walk that closes furthest above its earlier high ("breakout"). The seed was chosen so all three are easy to see: that is the point. Noise draws patterns on its own.

In [ ]:
rng = np.random.default_rng(179)
p = 100 * np.exp(np.cumsum(rng.normal(0, 0.01, (6, 250)), axis=1))
x = np.arange(250)
slope = np.array([np.polyfit(x, row, 1)[0] for row in p])
up, slide = int(np.argmax(slope)), int(np.argmin(slope))
rest = [k for k in range(6) if k not in (up, slide)]
brk = max(rest, key=lambda k: p[k, -1] / p[k, :-20].max())
print("uptrend? panel", up, " breakout? panel", brk, " sliding? panel", slide)
print("final prices:", p[:, -1].round(1))

# How often does pure noise produce at least one 'pattern' this clear? Try 1,000 fresh grids of six.
hits = 0
for s in range(1000):
    q = 100 * np.exp(np.cumsum(np.random.default_rng(10_000 + s).normal(0, 0.01, (6, 250)), axis=1))
    hits += (q[:, -1] > 115).any() or (q[:, -1] < 85).any()
print(f"grids with a walk ending 15% up or down: {hits / 1000:.0%}")

## 2. How many decisions it takes to tell a hit rate from a coin
Each decision is right with probability p. Score +1 / −1: mean 2p − 1, standard deviation about 1. The t-statistic after n decisions is (2p − 1)·√n. Setting t = 2 gives n = 1 / (p − 0.5)².

In [ ]:
for p in (0.52, 0.55, 0.60):
    n = 1 / (p - 0.5) ** 2
    print(f"right {p:.0%} of the time: about {n:,.0f} decisions")

print(f"20 independent calls a year: 400 decisions take {400 / 20:.0f} years")
print(f"1 rule x 40 markets x 12 months: {1 * 40 * 12} decisions a year (fewer independent ones: markets move together)")

## 3. A drifting method never adds up
If the way you decide changes every 50 decisions, 400 decisions are 8 samples of 50. How often does a real 55% skill even look better than a coin in a sample of 50 (t ≥ 2)?

In [ ]:
rng = np.random.default_rng(1)
wins = rng.random((100_000, 50)) < 0.55
score = 2 * wins - 1
t = score.mean(axis=1) / (score.std(axis=1, ddof=1) / np.sqrt(50))
print(f"a real 55% edge clears t >= 2 in a sample of 50: {np.mean(t >= 2):.0%} of the time")
wins = rng.random((20_000, 400)) < 0.55
score = 2 * wins - 1
t = score.mean(axis=1) / (score.std(axis=1, ddof=1) / np.sqrt(400))
print(f"...and in one consistent sample of 400: {np.mean(t >= 2):.0%}")

## 4. A textbook rule: same data in, same decision out
Hold when the price is above its 12-month average; step aside when below. On a seeded monthly noise path (seed 5, 4% a month), run it twice and compare.

In [ ]:
price = 100 * np.exp(np.cumsum(np.random.default_rng(5).normal(0, 0.04, 180)))

def rule(price):
    avg = np.convolve(price, np.ones(12) / 12, "valid")
    return price[11:] > avg

a, b = rule(price), rule(price.copy())
print("decisions:", len(a), " identical on a re-run:", bool((a == b).all()), f" time in market: {a.mean():.0%}")

## 5. A thousand random rules on pure noise
1,000 random long/short rules on 10 years of monthly noise (seed 7). Nothing has an edge, so the middle of the pile sits at zero. The best one does not.

In [ ]:
rng = np.random.default_rng(7)
market = rng.normal(0, 1, 120)
signs = rng.choice([-1, 1], (1000, 120))
pnl = signs * market
sharpe = pnl.mean(axis=1) / pnl.std(axis=1) * np.sqrt(12)
print(f"median Sharpe {round(float(np.median(sharpe)), 2) + 0:+.2f}   best of 1,000: {sharpe.max():.2f}   top 10 average: {np.sort(sharpe)[-10:].mean():.2f}")
print(f"noise: a Sharpe estimate over 10 years has a standard error of about {1 / np.sqrt(10):.2f}")

### Try this
- Change the number of random rules to 10, 100, 10,000. How does the best one grow?
- Run the best rule from section 5 on a fresh noise path (a new seed). What happens to its Sharpe?

*Simulated results on synthetic data. Education only. Not financial advice.*